# Decision Tree & Random Forest Code Showcase  
## Predicting High-Income Potential with Tree-Based Machine Learning

This notebook isolates **my modeling contribution** from the team machine learning project.

The goal is to show the code workflow from:

1. Baseline Decision Tree with raw CPS variables  
2. Progressive preprocessing and feature grouping  
3. Pre-pruning through `max_depth` tuning  
4. Post-pruning through cost-complexity pruning  
5. Random Forest tuning and model comparison  
6. Visualization of model performance and feature importance  

> Run note: the full pipeline expects the original CPS ASEC file `pppub24.csv` in the same directory as this notebook.  
> This file was not included in the portfolio export, so the notebook is structured as a code showcase and can be rerun after adding the raw dataset.

## 0. Setup

The code below uses the same libraries as the original project notebooks: `pandas`, `numpy`, `sklearn`, `matplotlib`, and `seaborn`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

sns.set_theme(style="whitegrid")

DATA_PATH = Path("pppub24.csv")
print("Expected raw data path:", DATA_PATH.resolve())

## 1. Load Raw CPS ASEC Data

This project used the 2024 CPS ASEC dataset. The raw file contains coded Census variables, so the first step is to select and rename the variables needed for income classification.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print("Raw shape:", df_raw.shape)
print(list(df_raw.columns)[:20])

## 2. Baseline Decision Tree — No Preprocessing

This baseline uses raw CPS-coded variables directly.  
The purpose is to establish a reference point and observe whether the model is over-relying on a single variable before feature cleaning.

In [ ]:
rename_raw = {
    "A_AGE": "age",
    "A_SEX": "sex",
    "A_MARITL": "marital_status",
    "A_HGA": "education_level",
    "PRDTRACE": "race",
    "A_HRS1": "hours_per_week",
    "A_MJIND": "industry_code",
    "PENATVTY": "nativity_country"
}

raw_vars = [
    "A_AGE",
    "A_SEX",
    "A_MARITL",
    "A_HGA",
    "PRDTRACE",
    "A_HRS1",
    "A_MJIND",
    "PENATVTY"
]

# Binary target: 1 = income > $50K, 0 = income <= $50K
df_raw["target_over_50k"] = (df_raw["PTOTVAL"] > 50000).astype(int)

dfA = df_raw[raw_vars + ["target_over_50k"]].copy()
dfA = dfA.rename(columns=rename_raw)

raw_vars_renamed = [
    "age",
    "sex",
    "marital_status",
    "education_level",
    "race",
    "hours_per_week",
    "industry_code",
    "nativity_country"
]

X_A = dfA[raw_vars_renamed]
y_A = dfA["target_over_50k"]

X_A_train, X_A_test, y_A_train, y_A_test = train_test_split(
    X_A,
    y_A,
    test_size=0.2,
    random_state=42
)

tree_no_pre = DecisionTreeClassifier(
    max_depth=7,
    random_state=42
)

tree_no_pre.fit(X_A_train, y_A_train)
preds = tree_no_pre.predict(X_A_test)

print("\n=== No Preprocessing Model Evaluation ===")
print("Accuracy:", accuracy_score(y_A_test, preds))
print("Precision:", precision_score(y_A_test, preds))
print("Recall:", recall_score(y_A_test, preds))
print("F1 Score:", f1_score(y_A_test, preds))

In [ ]:
importances = tree_no_pre.feature_importances_

imp_no = pd.DataFrame({
    "feature": X_A.columns,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\n=== Feature Importance: No Preprocessing Model ===")
display(imp_no)

In [ ]:
results = pd.DataFrame(columns=[
    "model", "accuracy", "recall", "precision", "f1", "key_features", "importance"
])

importance_series = pd.Series(tree_no_pre.feature_importances_, index=X_A_train.columns)

acc = accuracy_score(y_A_test, preds)
pre = precision_score(y_A_test, preds)
recall = recall_score(y_A_test, preds)
f1 = f1_score(y_A_test, preds)

key_feat = importance_series.abs().idxmax()
key_importance = importance_series[key_feat]

results.loc[len(results)] = [
    "Decision Tree raw data",
    acc,
    recall,
    pre,
    f1,
    key_feat,
    key_importance
]

display(results)

In [ ]:
plt.figure(figsize=(22, 12))
plot_tree(
    tree_no_pre,
    feature_names=X_A.columns,
    class_names=["<=50k", ">50k"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Baseline Decision Tree — Raw CPS Variables")
plt.show()

## 3. Preprocessing Step 1 — Remove Noise and Rename Variables

I filtered the dataset to active working adults and removed records that were not meaningful for income modeling:

- age > 16  
- total income > 100  
- hours worked per week > 0  

This step shifts the model from raw population-level data toward the active workforce population.

In [ ]:
df0 = df_raw.copy()

df0 = df0[
    (df0["A_AGE"] > 16) &
    (df0["PTOTVAL"] > 100) &
    (df0["A_HRS1"] > 0)
]

rename_map = {
    "PERIDNUM": "person_id",
    "A_AGE": "age",
    "A_SEX": "sex",
    "A_MARITL": "marital_status",
    "PENATVTY": "nativity_country",
    "PRDTRACE": "race",
    "A_HGA": "education_level",
    "A_HRS1": "hours_per_week",
    "A_MJIND": "industry_code",
    "PTOTVAL": "total_income",
}

df0 = df0.rename(columns=rename_map)
df0["target_over_50k"] = (df0["total_income"] > 50000).astype(int)

print("Filtered active workforce shape:", df0.shape)
df0.head()

In [ ]:
X0 = df0[raw_vars_renamed]
y0 = df0["target_over_50k"]

X0_train, X0_test, y0_train, y0_test = train_test_split(
    X0,
    y0,
    test_size=0.2,
    random_state=42
)

tree0 = DecisionTreeClassifier(
    max_depth=7,
    random_state=42
)

tree0.fit(X0_train, y0_train)
preds = tree0.predict(X0_test)

print("\n=== Tree 0: Noise Removed ===")
print("Accuracy:", accuracy_score(y0_test, preds))
print("Precision:", precision_score(y0_test, preds))
print("Recall:", recall_score(y0_test, preds))
print("F1 Score:", f1_score(y0_test, preds))

importance_series = pd.Series(tree0.feature_importances_, index=X0_train.columns)
key_feat = importance_series.abs().idxmax()
key_importance = importance_series[key_feat]

results.loc[len(results)] = [
    "Decision Tree remove noise",
    accuracy_score(y0_test, preds),
    recall_score(y0_test, preds),
    precision_score(y0_test, preds),
    f1_score(y0_test, preds),
    key_feat,
    key_importance
]

display(results)

## 4. Preprocessing Step 2 — Group Categorical Variables

The original CPS variables include detailed category codes. For tree-based models, high-cardinality categories can create overly specific branches. I grouped categories into broader, more stable buckets.

The main transformations are:

- Marital status → single / married / widowed / divorced-separated  
- Nativity → top 5 most frequent countries, all others as `Other`  
- Race → six broader groups  
- Education → four ordinal tiers

In [ ]:
# Marital status grouping
map_marital = {
    7: 0,                 # Single / never married
    1: 1, 2: 1, 3: 1, 6: 1, # Married / spouse-related statuses
    4: 2,                 # Widowed
    5: 3                  # Divorced
}

dfB = df0.copy()
dfB["marital_status"] = dfB["marital_status"].map(map_marital)

X_B = dfB[raw_vars_renamed]
y_B = dfB["target_over_50k"]

X_B_train, X_B_test, y_B_train, y_B_test = train_test_split(
    X_B,
    y_B,
    test_size=0.2,
    random_state=42
)

tree1 = DecisionTreeClassifier(max_depth=7, random_state=42)
tree1.fit(X_B_train, y_B_train)
preds = tree1.predict(X_B_test)

print("\n=== Tree 1: Marital Status Grouped ===")
print("Accuracy:", accuracy_score(y_B_test, preds))
print("Precision:", precision_score(y_B_test, preds))
print("Recall:", recall_score(y_B_test, preds))
print("F1 Score:", f1_score(y_B_test, preds))

importance_series = pd.Series(tree1.feature_importances_, index=X_B_train.columns)
key_feat = importance_series.abs().idxmax()
key_importance = importance_series[key_feat]

results.loc[len(results)] = [
    "Decision Tree marital modified",
    accuracy_score(y_B_test, preds),
    recall_score(y_B_test, preds),
    precision_score(y_B_test, preds),
    f1_score(y_B_test, preds),
    key_feat,
    key_importance
]

display(results)

In [ ]:
# Nativity grouping: keep top N countries and group the rest as 0 = Other
dfC = dfB.copy()

N = 5
top_countries = dfC["nativity_country"].value_counts().nlargest(N).index

dfC["nativity_grouped"] = dfC["nativity_country"].where(
    dfC["nativity_country"].isin(top_countries),
    other=0
)

X_C = dfC[[
    "age", "sex", "marital_status", "education_level", "race",
    "hours_per_week", "industry_code", "nativity_grouped"
]]
y_C = dfC["target_over_50k"]

X_C_train, X_C_test, y_C_train, y_C_test = train_test_split(
    X_C,
    y_C,
    test_size=0.2,
    random_state=42
)

tree2 = DecisionTreeClassifier(max_depth=7, random_state=42)
tree2.fit(X_C_train, y_C_train)
preds = tree2.predict(X_C_test)

print("\n=== Tree 2: Nativity Grouped ===")
print("Accuracy:", accuracy_score(y_C_test, preds))
print("Precision:", precision_score(y_C_test, preds))
print("Recall:", recall_score(y_C_test, preds))
print("F1 Score:", f1_score(y_C_test, preds))

importance_series = pd.Series(tree2.feature_importances_, index=X_C_train.columns)
key_feat = importance_series.abs().idxmax()
key_importance = importance_series[key_feat]

results.loc[len(results)] = [
    "Decision Tree country grouped",
    accuracy_score(y_C_test, preds),
    recall_score(y_C_test, preds),
    precision_score(y_C_test, preds),
    f1_score(y_C_test, preds),
    key_feat,
    key_importance
]

display(results)

In [ ]:
# Race grouping
dfD = dfC.copy()

race_simplified = []
for r in dfD["race"]:
    if r == 1:
        race_simplified.append(1)   # White
    elif r == 2:
        race_simplified.append(2)   # Black or African American
    elif r == 3:
        race_simplified.append(3)   # American Indian / Alaska Native
    elif r == 4:
        race_simplified.append(4)   # Asian
    elif r == 5:
        race_simplified.append(5)   # Native Hawaiian / Other Pacific Islander
    else:
        race_simplified.append(6)   # Two or more races / mixed

dfD["race_grouped"] = race_simplified

X_D = dfD[[
    "age", "sex", "marital_status", "education_level",
    "hours_per_week", "industry_code", "nativity_grouped", "race_grouped"
]]
y_D = dfD["target_over_50k"]

X_D_train, X_D_test, y_D_train, y_D_test = train_test_split(
    X_D,
    y_D,
    test_size=0.2,
    random_state=42
)

tree3 = DecisionTreeClassifier(max_depth=7, random_state=42)
tree3.fit(X_D_train, y_D_train)
preds = tree3.predict(X_D_test)

print("\n=== Tree 3: Race Grouped ===")
print("Accuracy:", accuracy_score(y_D_test, preds))
print("Precision:", precision_score(y_D_test, preds))
print("Recall:", recall_score(y_D_test, preds))
print("F1 Score:", f1_score(y_D_test, preds))

importance_series = pd.Series(tree3.feature_importances_, index=X_D_train.columns)
key_feat = importance_series.abs().idxmax()
key_importance = importance_series[key_feat]

results.loc[len(results)] = [
    "Decision Tree race grouped",
    accuracy_score(y_D_test, preds),
    recall_score(y_D_test, preds),
    precision_score(y_D_test, preds),
    f1_score(y_D_test, preds),
    key_feat,
    key_importance
]

display(results)

In [ ]:
# Education grouping
dfE = dfD.copy()

edu_grouped = []
for e in dfE["education_level"]:
    if 31 <= e <= 42:
        edu_grouped.append(0)   # Below bachelor's
    elif e == 43:
        edu_grouped.append(1)   # Bachelor's
    elif e == 44:
        edu_grouped.append(2)   # Master's
    elif 45 <= e <= 46:
        edu_grouped.append(3)   # Doctorate / professional degree
    else:
        edu_grouped.append(pd.NA)

dfE["education_grouped"] = edu_grouped

X_e = dfE[[
    "age", "sex", "marital_status", "education_grouped",
    "hours_per_week", "industry_code", "nativity_grouped", "race_grouped"
]]
y_e = dfE["target_over_50k"]

X_e_train, X_e_test, y_e_train, y_e_test = train_test_split(
    X_e,
    y_e,
    test_size=0.2,
    random_state=42
)

tree4 = DecisionTreeClassifier(max_depth=7, random_state=42)
tree4.fit(X_e_train, y_e_train)
preds = tree4.predict(X_e_test)

print("\n=== Tree 4: Education Grouped ===")
print("Accuracy:", accuracy_score(y_e_test, preds))
print("Precision:", precision_score(y_e_test, preds))
print("Recall:", recall_score(y_e_test, preds))
print("F1 Score:", f1_score(y_e_test, preds))

importance_series = pd.Series(tree4.feature_importances_, index=X_e_train.columns)
key_feat = importance_series.abs().idxmax()
key_importance = importance_series[key_feat]

results.loc[len(results)] = [
    "Decision Tree education grouped",
    accuracy_score(y_e_test, preds),
    recall_score(y_e_test, preds),
    precision_score(y_e_test, preds),
    f1_score(y_e_test, preds),
    key_feat,
    key_importance
]

display(results)

In [ ]:
# Visualize metric movement across preprocessing stages
results_plot = results.copy()
results_plot = results_plot.set_index("model")[["accuracy", "recall", "precision", "f1"]]

ax = results_plot.plot(figsize=(12, 6), marker="o")
ax.set_title("Decision Tree Performance Across Preprocessing Stages")
ax.set_xlabel("Model Version")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Final Preprocessed Decision Tree

This is the final grouped-feature version used for the Decision Tree comparison.

In [ ]:
final_col = [
    "person_id",
    "age",
    "sex",
    "marital_status",
    "nativity_grouped",
    "race_grouped",
    "education_grouped",
    "hours_per_week",
    "industry_code",
    "total_income",
    "target_over_50k"
]

df_clean_final = dfE[final_col].copy()

numeric_cols = [
    "age",
    "hours_per_week",
    "total_income",
    "industry_code",
    "target_over_50k",
    "sex",
    "marital_status",
    "education_grouped",
    "race_grouped",
    "nativity_grouped"
]

for col in numeric_cols:
    df_clean_final[col] = pd.to_numeric(df_clean_final[col], errors="coerce")

df_clean_final = df_clean_final.dropna(subset=[
    "target_over_50k",
    "age",
    "sex",
    "marital_status",
    "nativity_grouped",
    "race_grouped",
    "education_grouped",
    "hours_per_week",
    "industry_code"
])

df_clean_final["target_over_50k"] = df_clean_final["target_over_50k"].astype(int)

df_clean_final["income_group"] = df_clean_final["target_over_50k"].map({
    0: "≤ $50K",
    1: "> $50K"
})

FEATURES = [
    "age",
    "sex",
    "marital_status",
    "nativity_grouped",
    "race_grouped",
    "education_grouped",
    "hours_per_week",
    "industry_code"
]

X = df_clean_final[FEATURES]
y = df_clean_final["target_over_50k"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tree = DecisionTreeClassifier(
    max_depth=7,
    random_state=42
)

tree.fit(X_train, y_train)
preds = tree.predict(X_test)

print("\n=== Final Preprocessed Decision Tree ===")
print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))
print("F1 Score:", f1_score(y_test, preds))

imp_pre = pd.DataFrame({
    "feature": X.columns,
    "importance": tree.feature_importances_
}).sort_values("importance", ascending=False)

display(imp_pre)

In [ ]:
plt.figure(figsize=(22, 12))
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=["<=50k", ">50k"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Final Preprocessed Decision Tree")
plt.show()

## 6. Dummy Encoding Check

I also tested a dummy-encoded version to see whether expanding categorical variables into binary indicators improved Decision Tree performance.  
The result was useful because it showed that one-hot expansion can fragment the tree structure and redistribute feature importance.

In [ ]:
df_model = df_clean_final[[
    "age",
    "sex",
    "marital_status",
    "nativity_grouped",
    "race_grouped",
    "education_grouped",
    "hours_per_week",
    "industry_code",
    "target_over_50k"
]]

cols_to_dummy = [
    "sex",
    "marital_status",
    "nativity_grouped",
    "race_grouped",
    "education_grouped",
    "industry_code"
]

df_dummy = pd.get_dummies(df_model, columns=cols_to_dummy, drop_first=True)

X_dummy = df_dummy.drop(columns=["target_over_50k"])
y_dummy = df_dummy["target_over_50k"]

X_dummy_train, X_dummy_test, y_dummy_train, y_dummy_test = train_test_split(
    X_dummy,
    y_dummy,
    test_size=0.2,
    random_state=42,
    stratify=y_dummy
)

tree5 = DecisionTreeClassifier(
    max_depth=7,
    random_state=42
)

tree5.fit(X_dummy_train, y_dummy_train)
preds = tree5.predict(X_dummy_test)

print("\n=== Decision Tree with Dummy Variables ===")
print("Accuracy:", accuracy_score(y_dummy_test, preds))
print("Precision:", precision_score(y_dummy_test, preds))
print("Recall:", recall_score(y_dummy_test, preds))
print("F1 Score:", f1_score(y_dummy_test, preds))

importance_series = pd.Series(tree5.feature_importances_, index=X_dummy_train.columns)
key_feat = importance_series.abs().idxmax()
key_importance = importance_series[key_feat]

results.loc[len(results)] = [
    "Decision Tree with dummy",
    accuracy_score(y_dummy_test, preds),
    recall_score(y_dummy_test, preds),
    precision_score(y_dummy_test, preds),
    f1_score(y_dummy_test, preds),
    key_feat,
    key_importance
]

display(results)

## 7. Pre-Pruning — Decision Tree Depth Tuning

Here I compared several tree depths to illustrate the bias-variance trade-off:

- `max_depth=3`: simpler, more interpretable, but likely underfits  
- `max_depth=5`: moderate complexity  
- `max_depth=7`: best balance in the project result  
- `max_depth=None`: fully expanded tree, likely overfits

In [ ]:
depths = {
    "treeA": 3,
    "treeB": 5,
    "treeC": 7,
    "treeD": None
}

models = {}

for name, depth in depths.items():
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)
    models[name] = model
    print(f"{name} trained (max_depth={depth})")

In [ ]:
summary_rows = {}

for name, model in models.items():
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"\n=== {name} Evaluation ===")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)

    importances = model.feature_importances_
    top_idx = np.argmax(importances)

    summary_rows[name] = {
        "model": name,
        "max_depth": depths[name],
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "top_feature": X.columns[top_idx],
        "top_feature_importance": importances[top_idx]
    }

summary_df = pd.DataFrame(summary_rows).T
summary_df = summary_df.sort_values("f1_score", ascending=False)

print("\n=== Decision Tree Depth Tuning Summary ===")
display(summary_df)

In [ ]:
plt.figure(figsize=(10, 5))
summary_df[["accuracy", "precision", "recall", "f1_score"]].plot(kind="bar", figsize=(10, 5))
plt.title("Decision Tree Pre-Pruning: Metric Comparison by max_depth")
plt.xlabel("Tree Version")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the best pre-pruned tree from the original experiment: treeC, max_depth=7
plt.figure(figsize=(22, 12))
plot_tree(
    models["treeC"],
    feature_names=X.columns,
    class_names=["<=50k", ">50k"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree C — max_depth=7")
plt.show()

## 8. Post-Pruning — Cost-Complexity Pruning

After selecting Tree C (`max_depth=7`), I applied post-pruning with `ccp_alpha`.  
This reduces weak branches after the tree is grown and helps improve generalization.

In [ ]:
path = models["treeC"].cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]

alpha_scores = []

for ccp in ccp_alphas:
    clf = DecisionTreeClassifier(
        max_depth=7,
        random_state=42,
        ccp_alpha=ccp
    )
    scores = cross_val_score(clf, X_train, y_train, cv=5)
    alpha_scores.append(scores.mean())

best_alpha = ccp_alphas[np.argmax(alpha_scores)]
print("Best ccp_alpha for Tree C:", best_alpha)

treeC_pruned = DecisionTreeClassifier(
    max_depth=7,
    random_state=42,
    ccp_alpha=best_alpha
)

treeC_pruned.fit(X_train, y_train)

print("\n=== Original Tree C (max_depth=7) ===")
print("Train accuracy:", models["treeC"].score(X_train, y_train))
print("Test accuracy :", models["treeC"].score(X_test, y_test))
print("F1 Score      :", f1_score(y_test, models["treeC"].predict(X_test)))

print("\n=== Pruned Tree C (max_depth=7, post-pruned) ===")
print("Train accuracy:", treeC_pruned.score(X_train, y_train))
print("Test accuracy :", treeC_pruned.score(X_test, y_test))
print("F1 Score      :", f1_score(y_test, treeC_pruned.predict(X_test)))

In [ ]:
pruning_comparison = pd.DataFrame([
    {
        "model": "Original Tree C",
        "train_accuracy": models["treeC"].score(X_train, y_train),
        "test_accuracy": models["treeC"].score(X_test, y_test),
        "f1_score": f1_score(y_test, models["treeC"].predict(X_test))
    },
    {
        "model": "Post-Pruned Tree C",
        "train_accuracy": treeC_pruned.score(X_train, y_train),
        "test_accuracy": treeC_pruned.score(X_test, y_test),
        "f1_score": f1_score(y_test, treeC_pruned.predict(X_test))
    }
])

display(pruning_comparison)

ax = pruning_comparison.set_index("model")[["train_accuracy", "test_accuracy", "f1_score"]].plot(
    kind="bar",
    figsize=(8, 5)
)
ax.set_title("Pre-Pruning vs Post-Pruning Performance")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fi_pruned = pd.DataFrame({
    "feature": X.columns,
    "importance": treeC_pruned.feature_importances_
}).sort_values("importance", ascending=False)

print("=== Feature Importance: Tree C Post-Pruned ===")
display(fi_pruned)

plt.figure(figsize=(8, 5))
plt.barh(fi_pruned["feature"], fi_pruned["importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance — Post-Pruned Decision Tree")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(25, 15))

plot_tree(
    treeC_pruned,
    feature_names=X.columns,
    class_names=["<=50K", ">50K"],
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Decision Tree C — max_depth=7, post-pruned", fontsize=16)
plt.show()

## 9. Random Forest — Ensemble Tree Model

Random Forest extends Decision Trees by training multiple trees on bootstrapped samples and averaging their predictions.  
Here I tuned two key hyperparameters:

- `max_depth`: tree complexity  
- `n_estimators`: number of trees in the forest

In [ ]:
rf_param_list = []

for depth in [3, 5, 7, 9]:
    for est in [100, 200, 300, 500]:
        name = f"rf_d{depth}_n{est}"
        rf_param_list.append((name, depth, est))

rf_models = {}
rf_summary = {}

for name, depth, est in rf_param_list:
    model = RandomForestClassifier(
        n_estimators=est,
        max_depth=depth,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    rf_models[name] = model
    print(f"{name} trained → max_depth={depth}, n_estimators={est}")

In [ ]:
for name, depth, est in rf_param_list:
    model = rf_models[name]
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    importances = model.feature_importances_
    top_idx = np.argmax(importances)

    rf_summary[name] = {
        "model": name,
        "max_depth": depth,
        "n_estimators": est,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "top_feature": X.columns[top_idx],
        "top_feature_importance": importances[top_idx]
    }

rf_summary_df = pd.DataFrame(rf_summary).T
rf_summary_df = rf_summary_df.sort_values("f1_score", ascending=False)

print("\n=== Final Random Forest Summary ===")
display(rf_summary_df.head(10))

In [ ]:
best_rf_name = rf_summary_df.index[0]
best_rf = rf_models[best_rf_name]

print("Best Random Forest:", best_rf_name)
print(rf_summary_df.loc[best_rf_name])

rf_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": best_rf.feature_importances_
}).sort_values("importance", ascending=False)

display(rf_importance)

plt.figure(figsize=(8, 5))
plt.barh(rf_importance["feature"], rf_importance["importance"])
plt.gca().invert_yaxis()
plt.title(f"Feature Importance — Best Random Forest ({best_rf_name})")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 10. Decision Tree vs Random Forest Comparison

This final section compares:

- tuned Decision Trees  
- post-pruned Decision Tree  
- Random Forest models  

The main interpretation is that Random Forest can tolerate deeper trees because ensemble averaging reduces variance.

In [ ]:
summary_df["model_type"] = "Decision Tree"
rf_summary_df["model_type"] = "Random Forest"

compare_df = pd.concat([summary_df, rf_summary_df], axis=0)
compare_df_plot = compare_df[["model", "model_type", "accuracy", "f1_score"]].reset_index(drop=True)

display(compare_df_plot.head())

In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(data=compare_df_plot, x="model", y="accuracy", hue="model_type")

plt.title("Accuracy Comparison: Decision Tree vs Random Forest", fontsize=16)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Model Type")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(data=compare_df_plot, x="model", y="f1_score", hue="model_type")

plt.title("F1 Score Comparison: Decision Tree vs Random Forest", fontsize=16)
plt.xlabel("Model", fontsize=12)
plt.ylabel("F1 Score", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Model Type")
plt.tight_layout()
plt.show()

In [ ]:
dt_importance = pd.DataFrame({
    "feature": X.columns,
    "dt_importance": models["treeC"].feature_importances_
})

rf_importance_for_compare = pd.DataFrame({
    "feature": X.columns,
    "rf_importance": best_rf.feature_importances_
})

importance_compare = dt_importance.merge(rf_importance_for_compare, on="feature")
importance_compare = importance_compare.sort_values("rf_importance", ascending=False)

display(importance_compare)

ax = importance_compare.set_index("feature")[["dt_importance", "rf_importance"]].plot(
    kind="barh",
    figsize=(9, 6)
)
ax.set_title("Feature Importance Comparison: Decision Tree vs Random Forest")
ax.set_xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Portfolio Interpretation

This code track demonstrates that I did more than call a classifier once.  
I built a full tree-based modeling workflow:

- established a raw Decision Tree baseline  
- diagnosed over-reliance on a single feature  
- progressively engineered grouped categorical features  
- tuned Decision Tree depth to manage bias and variance  
- applied post-pruning with cost-complexity pruning  
- compared single-tree logic against Random Forest ensemble performance  
- used feature importance and visualizations to explain why the ensemble model generalized better  

This is the part of the project I would emphasize in interviews when discussing nonlinear models, overfitting control, and model interpretability.